# Performance Ranking & Outlier Analysis
## Domain: Performance Leaderboards & Pareto Ranking
### Objectives & Methodology:
- Identify top-performing and underperforming products and customer cohorts.
- Rank products by gross revenue using both ANSI `LIMIT` and window functions (`ROW_NUMBER()`).
- Pinpoint laggard items with minimal sales to inform inventory rationalization.
- Rank high-value VIP customers driving disproportionate store revenue.
- Identify infrequent or single-purchase customer accounts to target with re-engagement incentives.


In [ ]:
# Databricks notebook source
# Setup environment and display helper
import os
from pyspark.sql import SparkSession

try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)


## 1. Top 5 Revenue-Generating Products
Rank the top 5 highest-grossing products using ANSI `ORDER BY ... LIMIT 5`.


In [ ]:
# Top 5 products generating the highest revenue
df_top5_products = spark.sql("""
SELECT
    p.product_name,
    SUM(f.sales_amount) AS total_revenue
FROM gold.fact_sales f
LEFT JOIN gold.dim_products p ON p.product_key = f.product_key
GROUP BY p.product_name
ORDER BY total_revenue DESC
LIMIT 5
""")
display(df_top5_products)


## 2. Top 5 Products Using Window Ranking Functions
Apply `ROW_NUMBER() OVER(ORDER BY SUM(sales_amount) DESC)` for flexible partitioned and windowed ranking.


In [ ]:
# Complex but flexible ranking using window functions (Top 5)
df_top5_window = spark.sql("""
SELECT
    p.product_name,
    SUM(f.sales_amount) AS total_revenue,
    ROW_NUMBER() OVER(ORDER BY SUM(f.sales_amount) DESC) AS rank_products
FROM gold.fact_sales f
LEFT JOIN gold.dim_products p ON p.product_key = f.product_key
GROUP BY p.product_name
LIMIT 5
""")
display(df_top5_window)


## 3. Bottom 5 Laggard Products by Sales Volume
Identify the 5 lowest-grossing products to highlight slow-moving inventory.


In [ ]:
# Bottom 5 worst performing products in terms of sales
df_bottom5_products = spark.sql("""
SELECT
    p.product_name,
    SUM(f.sales_amount) AS total_revenue
FROM gold.fact_sales f
LEFT JOIN gold.dim_products p ON p.product_key = f.product_key
GROUP BY p.product_name
ORDER BY total_revenue ASC
LIMIT 5
""")
display(df_bottom5_products)


## 4. Top 10 High-Value Customers (VIP Leaderboard)
Identify the top 10 customer patrons generating the highest cumulative revenue.


In [ ]:
# Top 10 customers generating the highest revenue
df_top10_customers = spark.sql("""
SELECT
    c.customer_key,
    CONCAT(c.first_name, ' ', c.last_name) AS full_name,
    SUM(f.sales_amount) AS total_revenue
FROM gold.fact_sales f
LEFT JOIN gold.dim_customers c ON c.customer_key = f.customer_key
GROUP BY c.customer_key, CONCAT(c.first_name, ' ', c.last_name)
ORDER BY total_revenue DESC
LIMIT 10
""")
display(df_top10_customers)


## 5. Infrequent Purchasers: Bottom 3 Customers by Order Frequency
Identify customer accounts with minimal transaction counts for targeted retention campaigns.


In [ ]:
# The 3 customers with the fewest orders placed
df_bottom3_customers = spark.sql("""
SELECT
    c.customer_key,
    CONCAT(c.first_name, ' ', c.last_name) AS full_name,
    COUNT(DISTINCT f.order_number) AS total_orders
FROM gold.fact_sales f
LEFT JOIN gold.dim_customers c ON f.customer_key = c.customer_key
GROUP BY c.customer_key, CONCAT(c.first_name, ' ', c.last_name)
ORDER BY total_orders ASC
LIMIT 3
""")
display(df_bottom3_customers)


## Business Takeaways & Merchandising Insights
- **Flagship Models**: High-end Mountain and Road bikes consistently command top positions on the revenue leaderboard.
- **Inventory Rationalization**: Bottom-ranked products (e.g. low-volume replacement parts) should be reviewed for stock optimization or bundling.
- **VIP Customer Retention**: Top 10 VIP customers contribute disproportionately to profitability; proactive loyalty programs should safeguard their engagement.
